# Chapter 9 lab — What does a timeout tell us about a purchase?

Draft educator companion v2 · 8 September 2026 · 90 minutes. This version supersedes v1 for new classes; retain your old attempts.

**What we are building:** a useful agent for Lucy’s shop, whose actual Python boundaries can be investigated and repaired. **What Lucy loses if this boundary fails:** Lucy can buy the same replenishment twice while both local attempts remain uncertain.

Read [the chapter](https://www.profrod.ai/book/ch09-ambiguous-order). Prerequisites: Chapter 8 approvals; supplier receipt, local intent and uncertain outcomes. Use the locked Python 3.14 checkout named by the educator companion. Set SOVEREIGN_AGENT_REPO if the notebook is outside that checkout. No packages are installed here; no model or channel credentials are needed.

Today’s sequence: predict 10 minutes; decision warm-up 10; real-code break and repair 45; transfer 15; exit ticket 10. The warm-up is deliberately small. The central task edits a temporary copy of the actual implementation, observes its effect, repairs it, and traces a source line to a retained result.

**Execution preparation:** One probe subprocess launches one separate local HTTP supplier process on an ephemeral loopback port. The supplier retains SQLite receipts and drops first responses. Both processes and temporary state are cleaned up; no live purchase or SIGKILL. Allow up to 60 seconds per trial for a cold machine; measure elapsed classroom time yourself. Copied Python is not a security sandbox. Run only these reviewed local exercises, never arbitrary downloaded code. Close the lab at the end to remove its temporary copy; save your source patch and evidence first.


## Predict before running

The supplier accepted vanilla but the response disappeared. The local ledger says UNKNOWN. Is it safe to create a new operation ID and try again?

Write a prediction for the real mutation too: receipt = supplier.order(identifier, json.loads(row["proposal"])) → receipt = supplier.order(uuid.uuid4().hex, json.loads(row["proposal"])). Name an observation that would disprove your explanation.


In [ ]:
import copy
import hashlib
import json
import os
import runpy
import subprocess
import sys
from pathlib import Path

if sys.version_info < (3, 14):
    raise RuntimeError("Use the book Python 3.14 environment for Chapters 2–16.")
# Open the notebook inside your source checkout, or set this path explicitly.
start = Path(os.environ.get("SOVEREIGN_AGENT_REPO", Path.cwd())).resolve()
ROOT = next(
    (p for p in (start, *start.parents) if (p / "book/always_on/checkpoints/ch09.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Set SOVEREIGN_AGENT_REPO to the Sovereign Agent checkout.")
CHECKPOINT = ROOT / "book/always_on/checkpoints/ch09.py"
EXPECTED_CHECKPOINT_SHA256 = "1993459a746158cb289821f5980d9c4e365fc93ea1665eb96fce66323f5bae96"
if hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest() != EXPECTED_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Checkpoint differs; use the exact cohort commit and versioned files in the companion."
    )
print("Chapter 9 checkpoint bytes match this lesson. No model or channel has been called.")

lab_support = runpy.run_path(str(ROOT / "book/always_on/educator/runtime_labs_v1.py"))
RuntimeLab = lab_support["RuntimeLab"]

## Ten-minute decision warm-up

Implement decide(case). A validated receipt must match operation; a mismatch returns REFUSED. Matching ACCEPTED means SETTLE, matching REJECTED means RELEASE. No receipt (None) or any nonconclusive status means HOLD. Input receipts here are already authenticated and schema-checked; this is only the next-step classifier.

Visible tests are a specification, not secret examination questions. A lookup table of CASES can pass them. Your teacher assesses your implementation, a novel teacher-chosen case, and the actual runtime repair. Do not infer mastery from the worked answer or from running supplied code. Input mutation is a failure even when expected and observed values match.


In [ ]:
def grade(candidate, cases):
    results = []
    for index, (case, expected) in enumerate(cases, 1):
        supplied = copy.deepcopy(case)
        row = {"case": index, "expected": expected, "mutated": False}
        try:
            observed = candidate(supplied)
            row["mutated"] = supplied != case
            try:
                encoded = json.dumps(observed, allow_nan=False, sort_keys=True)
                safe = json.loads(encoded)
            except (TypeError, ValueError, OverflowError, RecursionError):
                row.update(
                    status="FAILED",
                    reason="unsupported_return_type",
                    observed={
                        "result_type": type(observed).__name__,
                        "reason": "unsupported_return_type",
                    },
                )
            else:
                same = encoded == json.dumps(expected, allow_nan=False, sort_keys=True)
                row.update(
                    status="PASS" if same and not row["mutated"] else "FAILED",
                    reason=(
                        "input_mutated"
                        if row["mutated"]
                        else "matched_contract"
                        if same
                        else "wrong_result"
                    ),
                    observed=safe,
                )
        except NotImplementedError:
            row.update(status="NOT_SUBMITTED", reason="not_submitted", mutated=supplied != case)
        except Exception as error:
            row.update(
                status="FAILED",
                reason="candidate_exception",
                error_type=type(error).__name__,
                mutated=supplied != case,
            )
        results.append(row)
    return results


def assessment_status(results):
    counts = {
        name: sum(row["status"] == name for row in results)
        for name in ("PASS", "FAILED", "NOT_SUBMITTED")
    }
    status = (
        "NOT_SUBMITTED"
        if not results or counts["NOT_SUBMITTED"] == len(results)
        else "PARTIAL"
        if counts["NOT_SUBMITTED"]
        else "FAILED"
        if counts["FAILED"]
        else "PASSED_VISIBLE_CONTRACT"
    )
    return {
        "status": status,
        "counts": counts,
        "scope": "Visible cases need teacher review of code, novel cases and runtime evidence.",
    }


def decide(case):
    raise NotImplementedError("Write your function before consulting the worked solution.")

In [ ]:
CASES = [
    ({"operation": "vanilla-1", "receipt": None}, "HOLD"),
    (
        {"operation": "vanilla-1", "receipt": {"operation": "vanilla-1", "status": "ACCEPTED"}},
        "SETTLE",
    ),
    (
        {"operation": "vanilla-1", "receipt": {"operation": "vanilla-1", "status": "REJECTED"}},
        "RELEASE",
    ),
    (
        {"operation": "vanilla-1", "receipt": {"operation": "other", "status": "ACCEPTED"}},
        "REFUSED",
    ),
    (
        {"operation": "vanilla-1", "receipt": {"operation": "vanilla-1", "status": "PENDING"}},
        "HOLD",
    ),
    ({"operation": "vanilla-1", "receipt": {"operation": "other", "status": "PENDING"}}, "REFUSED"),
    (
        {"operation": "vanilla-1", "receipt": {"operation": "vanilla-1", "status": "TIMEOUT"}},
        "HOLD",
    ),
]
submission_results = grade(decide, CASES)
submission_summary = assessment_status(submission_results)
print(json.dumps({"summary": submission_summary, "cases": submission_results}, indent=2))

## Forty-five-minute centre — break and repair the real implementation

Target `src/sovereign_agent/assistant_orders.py` at the line printed below. Read the surrounding function and the probe before executing it. The helper copies actual source, learner code, checkpoints and skills into a new temporary directory. It verifies the original file fingerprint and never edits the checkout.

The probe is plain Python, visible at `lab.probe`; open it and follow its inputs into the target function. It uses real tool dispatch or database operations, not a second toy implementation. The literal expected observations were authored separately from the code under test.

The broken send invents a new ID, so lookup of the original operation cannot find either purchase. Moving lookup after a resend would not itself prove duplication under this supplier’s same-ID idempotency contract. Retain that counterexample in the explanation.

Record baseline and broken observations. Before revealing the answer, replace REPAIR_FRAGMENT with your repair of the marked fragment, or edit `lab.target` directly and set EDITED_COPY=True. A blank repair remains NOT_SUBMITTED. Do not edit installed runtime code, the original checkout, or the printed expected answers.


In [ ]:
lab = RuntimeLab(ROOT, 9)
print(str(lab.target))
print(lab.source_excerpt())
print(lab.probe.read_text())
baseline_record = lab.run("BASELINE_REFERENCE", expected=lab.spec["expected_baseline"])
lab.break_source()
print(lab.source_excerpt())
broken_record = lab.run("BROKEN_REFERENCE", expected=lab.spec["expected_broken"])
# PASS here means the supplied fault produced the expected failure, not that it is safe.
assert baseline_record["status"] == "PASS", "Inspect retained baseline diagnostics"
assert broken_record["status"] == "PASS", "Inspect retained fault diagnostics"

In [ ]:
REPAIR_FRAGMENT = None  # TODO: replacement text for the marked broken fragment.
EDITED_COPY = False  # Set True only after saving your own edit to lab.target.
if REPAIR_FRAGMENT is not None:
    lab.repair(REPAIR_FRAGMENT)
if REPAIR_FRAGMENT is not None or EDITED_COPY:
    student_repair = lab.run("STUDENT_REPAIR", expected=lab.spec["expected_baseline"])
    student_source = lab.target.read_text()
else:
    student_repair = {"status": "NOT_SUBMITTED"}
    student_source = None
TRACE = {}  # TODO: source_path, source_line, observation_key, observed_value, explanation.
trace_result = lab.trace(TRACE, broken_record)
print(json.dumps({"repair": student_repair, "trace": trace_result}, indent=2))

## Transfer — change one constraint

Add an absent lookup result after a process restart and keep it HOLD. Trace why execute can query/reconcile the same operation while the local ledger is reopened. What extra provider contract makes a repeat send safe?

Add independently calculated (input, expected) pairs below. For the runtime extension, edit only the copied probe and retain a new trial. Do not feed runtime objects into a pure-function case.

**Real-code extension:** After repairing, directly repeat an identical supplier request using the same operation and proposal; inspect independent rows. Then explain why absent discovery without idempotency still requires HOLD.


In [ ]:
TRANSFER_CASES = []
transfer_results = grade(decide, TRANSFER_CASES)
transfer_summary = assessment_status(transfer_results)
print(json.dumps({"summary": transfer_summary, "cases": transfer_results}, indent=2))

## Optional whole-chapter checkpoint

This separate reference retains the complete integration scenario. It is not your repair grade. Set RUN_FULL_CHECKPOINT=True only after the central experiment and with time to inspect it. Chapter 10 and 16 include real local SIGKILL; Chapters 9–10 and 15–16 start supplier processes; Chapter 11 starts MCP; Chapter 15 never installs systemd. Failures print stdout and stderr before any assertion.


In [ ]:
RUN_FULL_CHECKPOINT = False
if RUN_FULL_CHECKPOINT:
    # This supplied cumulative program is separate from grading your function.
    # It uses fixture models/channels. Some chapters start local child processes.
    # No --live, --telegram or --containers switch is added.
    reference_environment = {
        k: v
        for k, v in os.environ.items()
        if k in {"PATH", "SYSTEMROOT", "TMPDIR", "LANG", "LC_ALL"}
    }
    reference_environment["PYTHONPATH"] = str(ROOT / "src")
    reference_run = subprocess.run(
        [sys.executable, str(CHECKPOINT)],
        cwd=ROOT,
        env=reference_environment,
        capture_output=True,
        text=True,
        timeout=180,
        check=False,
    )
    print(reference_run.stdout)
    print(reference_run.stderr, file=sys.stderr)
    assert reference_run.returncode == 0, "Checkpoint failed; stdout and stderr are retained above"
    EXPECTED_OBSERVATIONS = ["initial UNKNOWN", "reserved and spent 1500 0", "supplier orders 1"]
    assert all(text in reference_run.stdout for text in EXPECTED_OBSERVATIONS)
    print("REFERENCE_CHECKPOINT_PASSED — this is not your submission grade.")

## Worked answers — reveal after retaining your attempt

The supplier’s idempotency contract must bind the stable operation ID to the same proposal. Missing discovery is not evidence of rejection. Before reconciliation reservation remains 1500; afterwards it is spent 1500 / reserved 0, with exactly one independently observed supplier row in this fixture.

The broken send invents a new ID, so lookup of the original operation cannot find either purchase. Moving lookup after a resend would not itself prove duplication under this supplier’s same-ID idempotency contract. Retain that counterexample in the explanation.

Teacher trace key: `src/sovereign_agent/assistant_orders.py:390` → {'supplier_order_count': 1, 'local_statuses': ['UNKNOWN', 'ACCEPTED']} before the fault and {'supplier_order_count': 2, 'local_statuses': ['UNKNOWN', 'UNKNOWN']} after it. The source line changes the real probe result. Merely printing those values is not a repair.


In [ ]:
def worked_decide(case):
    receipt = case["receipt"]
    if receipt is None:
        return "HOLD"
    if receipt["operation"] != case["operation"]:
        return "REFUSED"
    return {"ACCEPTED": "SETTLE", "REJECTED": "RELEASE"}.get(receipt["status"], "HOLD")


worked_results = grade(worked_decide, CASES)
assert all(row["status"] == "PASS" for row in worked_results)
# A fresh copy keeps the worked repair separate from your retained source/evidence.
worked_lab = RuntimeLab(ROOT, 9)
try:
    worked_lab.break_source()
    worked_lab.repair(worked_lab.spec["before"])
    worked_repair = worked_lab.run("WORKED_REPAIR", expected=worked_lab.spec["expected_baseline"])
finally:
    worked_lab.close()
print("WORKED_EXAMPLE_PASSED; original submission remains separate.")
assert worked_repair["status"] == "PASS", "Worked repair failed; inspect diagnostics"

In [ ]:
def tempting_shortcut(case):
    return "RELEASE" if case["receipt"] is None else "SETTLE"


shortcut_results = grade(tempting_shortcut, CASES)
assert any(row["status"] == "FAILED" for row in shortcut_results)
print(json.dumps(shortcut_results, indent=2))

## Exit ticket and evidence

Submit the first attempt, warm-up cases, your repaired source, baseline/broken/repaired records, source-to-result trace, transfer prediction and outcome. Explain Lucy’s consequence, which observation would falsify your claim, and what the probe leaves unproved.

A local rollback cannot roll back a remote purchase. A timeout is not a negative receipt. The classifier does not authorize retries or validate receipts. The chapter’s supplier simulator supplies an explicit idempotency/discovery contract; do not infer universal exactly-once effects.

Use the companion pilot record to capture actual completion times and errors. No automated run is evidence of a student learning gain.


In [ ]:
runtime_results = {
    "baseline": baseline_record,
    "broken": broken_record,
    "student_repair": student_repair,
    "student_source": student_source,
    "trace": trace_result,
    "worked_repair": worked_repair,
}
EVIDENCE_PATH = None  # Optional new path; existing evidence is never overwritten.
if EVIDENCE_PATH is not None:
    with Path(EVIDENCE_PATH).open("x") as stream:
        json.dump(
            {
                "submission": submission_results,
                "transfer": transfer_results,
                "runtime": runtime_results,
            },
            stream,
            indent=2,
        )
lab.close()